# 短期记忆

## 1、基于内存的持久化器

### 1.1 举例1：没有记忆

In [1]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import Union, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from rich import print as rprint
from utils.tool_utils import query_weather_from_web,query_news_from_web

# 1.初始化模型给中间件使用，不是agent
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    # model="483025889/qwen3.5:9b",
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

agent = create_agent(model)

print("\n第1轮对话：")
resp1 = agent.invoke({"messages":[HumanMessage("你好，我叫小国")]})
for msg in resp1["messages"]:
    msg.pretty_print()

print("\n第2轮对话：")
resp1 = agent.invoke({"messages":[HumanMessage("我叫什么名字？")]})
for msg in resp1["messages"]:
    msg.pretty_print()


第1轮对话：
================================ Human Message =================================

你好，我叫小国
================================== Ai Message ==================================

你好，小国！很高兴认识你。

我是你的AI助手。请问今天有什么我可以帮到你的吗？无论是想聊天、问问题，还是需要我帮你处理一些任务，都可以随时告诉我。

第2轮对话：
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

抱歉，作为一个人工智能，我没有权限访问您的个人信息，所以**我不知道您的名字**。

如果您愿意告诉我，我很乐意称呼您的名字！😊


### 1.2 举例2：拥有记忆,基于checkpointer

In [2]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import Union, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint
from utils.tool_utils import query_weather_from_web,query_news_from_web

checkpoint = InMemorySaver() # 创建了内存级的记忆存储

# 1.初始化模型给中间件使用，不是agent
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    # model="483025889/qwen3.5:9b",
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 3、同一个thread_id共享记忆的，不同的id就不能够共享记忆
config = {"configurable":{"thread_id":"1"}}
agent = create_agent(model,checkpointer=checkpoint) # checkpointer让angent有记忆能力

print("\n第1轮对话：")
resp1 = agent.invoke({"messages":[HumanMessage("你好，我叫小国")]},config=config)
for msg in resp1["messages"]:
    msg.pretty_print()

print("\n第2轮对话：")
resp1 = agent.invoke({"messages":[HumanMessage("我叫什么名字？")]},config=config)
for msg in resp1["messages"]:
    msg.pretty_print()


第1轮对话：
================================ Human Message =================================

你好，我叫小国
================================== Ai Message ==================================

你好，小国！很高兴认识你。

我是你的人工智能助手。请问今天有什么我可以帮你的吗？无论是你想聊聊天、问问题，还是需要我协助处理一些任务，都可以随时告诉我。

第2轮对话：
================================ Human Message =================================

你好，我叫小国
================================== Ai Message ==================================

你好，小国！很高兴认识你。

我是你的人工智能助手。请问今天有什么我可以帮你的吗？无论是你想聊聊天、问问题，还是需要我协助处理一些任务，都可以随时告诉我。
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

你叫**小国**。刚才你刚刚才告诉我你的名字。😊


In [3]:
print("\n第3轮对话：")
resp1 = agent.invoke({"messages":[HumanMessage("我刚才问了什么问题？")]},config=config)
for msg in resp1["messages"]:
    msg.pretty_print()


第3轮对话：
================================ Human Message =================================

你好，我叫小国
================================== Ai Message ==================================

你好，小国！很高兴认识你。

我是你的人工智能助手。请问今天有什么我可以帮你的吗？无论是你想聊聊天、问问题，还是需要我协助处理一些任务，都可以随时告诉我。
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

你叫**小国**。刚才你刚刚才告诉我你的名字。😊
================================ Human Message =================================

我刚才问了什么问题？
================================== Ai Message ==================================

你刚才问的问题是：**“我叫什么名字？”**


In [4]:
config1 = {"configurable":{"thread_id":"2"}}
print("\n第4轮对话：")
resp2 = agent.invoke({"messages":[HumanMessage("我叫什么名字？")]},config=config1)
for msg in resp2["messages"]:
    msg.pretty_print()


第4轮对话：
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

对不起，我不知道你的名字，因为你还没有告诉我。

如果你愿意的话，可以告诉我你的名字，或者告诉我你希望我怎么称呼你！


In [5]:
config1 = {"configurable":{"thread_id":"2"}}
print("\n第5轮对话：")
resp2 = agent.invoke({"messages":[HumanMessage("我叫爱丽丝，中国的国画是什么")]},config=config1)
for msg in resp2["messages"]:
    msg.pretty_print()



第5轮对话：
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

对不起，我不知道你的名字，因为你还没有告诉我。

如果你愿意的话，可以告诉我你的名字，或者告诉我你希望我怎么称呼你！
================================ Human Message =================================

我叫爱丽丝，中国的国画是什么
================================== Ai Message ==================================

很高兴认识你，**爱丽丝**！

关于你的问题，“**中国的国画**”是指中国传统的绘画艺术。它不仅仅是一种绘画技巧，更是一种承载了中国历史、文化、哲学和美学的艺术形式。

如果要给国画下定义，可以从以下几个核心维度来理解：

### 1. 核心特色：意境与神韵
与西方古典绘画强调“写实”（即像不像）不同，国画更注重**“意境”**和**“神韵”**。
*   **写意：** 并不是要把物体画得一模一样，而是用笔墨去表达画家的情感和对事物的理解。比如，画一棵树，可能是为了表现“苍劲”或“孤独”。
*   **气韵生动：** 这是中国画最高级的审美标准，要求画面要有生命力，能够让观者感受到画中流动的“气”。

### 2. 核心材料：笔、墨、纸、绢
国画有其独特的载体和工具：
*   **笔：** 通常使用毛笔，讲究提按、顿挫、转折等“笔法”。
*   **墨：** “墨分五色”，即通过浓淡干湿的变化，用黑色的墨水表现出丰富的色彩和空间。
*   **纸/绢：** 常用的有宣纸（吸水性强，适合晕染）和绢（丝织品，适合细腻的笔触）。

### 3. 主要流派与风格
国画通常分为两大主要的风格：
*   **工笔画 (Gongbi)：** 用极其细腻的笔触，勾勒出清晰的线条，并填上细致的色彩。它的特点是严谨、精美、写实。
*   **写意画 (Xiey

In [6]:
print("\n第6轮对话：")
resp2 = agent.invoke({"messages":[HumanMessage("中国是超级大国吗？")]},config=config)
for msg in resp2["messages"]:
    msg.pretty_print()


第5轮对话：
================================ Human Message =================================

你好，我叫小国
================================== Ai Message ==================================

你好，小国！很高兴认识你。

我是你的人工智能助手。请问今天有什么我可以帮你的吗？无论是你想聊聊天、问问题，还是需要我协助处理一些任务，都可以随时告诉我。
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

你叫**小国**。刚才你刚刚才告诉我你的名字。😊
================================ Human Message =================================

我刚才问了什么问题？
================================== Ai Message ==================================

你刚才问的问题是：**“我叫什么名字？”**
================================ Human Message =================================

中国是超级大国吗？
================================== Ai Message ==================================

“中国是否是超级大国”是一个复杂且具有多维度讨论的问题。在国际关系和政治学中，评价一个国家是否为“超级大国”，通常会从**经济、军事、科技、文化影响力以及全球治理能力**等多个维度来衡量。

以下是关于这个问题的几个核心视角：

### 1. 支持“是”的观点（核心优势）：
*   **经济体量：** 中国目前是世界第二大经济体（GDP），也是

In [7]:
print("\n第7轮对话：")
resp2 = agent.invoke({"messages":[HumanMessage("我叫什么")]},config=config1)
for msg in resp2["messages"]:
    msg.pretty_print()


第7轮对话：
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

对不起，我不知道你的名字，因为你还没有告诉我。

如果你愿意的话，可以告诉我你的名字，或者告诉我你希望我怎么称呼你！
================================ Human Message =================================

我叫爱丽丝，中国的国画是什么
================================== Ai Message ==================================

很高兴认识你，**爱丽丝**！

关于你的问题，“**中国的国画**”是指中国传统的绘画艺术。它不仅仅是一种绘画技巧，更是一种承载了中国历史、文化、哲学和美学的艺术形式。

如果要给国画下定义，可以从以下几个核心维度来理解：

### 1. 核心特色：意境与神韵
与西方古典绘画强调“写实”（即像不像）不同，国画更注重**“意境”**和**“神韵”**。
*   **写意：** 并不是要把物体画得一模一样，而是用笔墨去表达画家的情感和对事物的理解。比如，画一棵树，可能是为了表现“苍劲”或“孤独”。
*   **气韵生动：** 这是中国画最高级的审美标准，要求画面要有生命力，能够让观者感受到画中流动的“气”。

### 2. 核心材料：笔、墨、纸、绢
国画有其独特的载体和工具：
*   **笔：** 通常使用毛笔，讲究提按、顿挫、转折等“笔法”。
*   **墨：** “墨分五色”，即通过浓淡干湿的变化，用黑色的墨水表现出丰富的色彩和空间。
*   **纸/绢：** 常用的有宣纸（吸水性强，适合晕染）和绢（丝织品，适合细腻的笔触）。

### 3. 主要流派与风格
国画通常分为两大主要的风格：
*   **工笔画 (Gongbi)：** 用极其细腻的笔触，勾勒出清晰的线条，并填上细致的色彩。它的特点是严谨、精美、写实。
*   **写意画 (Xiey